# Build Snowflake Feature Store with getML

## Prerequisites

In [1]:
import os
import subprocess
import sys

# Load environment variables from mise.toml
result = subprocess.run(
    ["mise", "env"],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if line.startswith("export "):
        # Parse: export VAR=value or export VAR='value'
        _, var_value = line.split(" ", 1)
        if "=" in var_value:
            name, value = var_value.split("=", 1)
            # Strip quotes if present
            value = value.strip("'\"")
            os.environ[name] = value

# Add project root to path for imports
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import getml
from snowflake.snowpark import Session
from data import SnowflakeSettings

In [3]:
getml.set_project('snowflake_feature_store')

Connected to project 'snowflake_feature_store'.

## Setup and Data Loading 

In [4]:
settings = SnowflakeSettings.from_env()

connection_params: dict[str, str | int] = {
    "account": settings.account,
    "user": settings.user,
    "password": settings.password.get_secret_value(),
    "role": settings.role,
    "warehouse": settings.warehouse,
    "database": settings.database,
    "schema": settings.schema_name,
}

session = Session.builder.configs(connection_params).create()
arrow_table = session.table("PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET").to_arrow()
weekly_sales_by_store = getml.DataFrame.from_arrow(arrow_table, name="weekly_sales_by_store")

DataFrame.to_arrow() is experimental since 1.28.0. Do not use it in production. 
/home/alex/projects/worktrees/getml-demo/60-create-initial-snowflake-notebook-5-sections/integration/snowflake/.venv/lib/python3.12/site-packages/getml/data/_io/arrow.py:371: UserWarning:     
Column 'NEXT_WEEK_SALES' has been converted from decimal to float. This may
    result in a loss of precision!
    
  warnings.warn(


## getml Annotations

In [14]:
weekly_sales_by_store

name,YEAR,MONTH,WEEK_NUMBER,SNAPSHOT_ID,DAYS_SINCE_OPEN,NEXT_WEEK_SALES,NEXT_WEEK_ORDERS,STORE_ID,STORE_NAME,REFERENCE_DATE,IS_FULL_WEEK_AFTER_OPENING,HAS_ORDER_ACTIVITY,HAS_MIN_HISTORY
role,categorical,categorical,categorical,unused_float,unused_float,unused_float,unused_float,unused_string,unused_string,unused_string,unused_string,unused_string,unused_string
0,2019,5,21,48,261,13165.49,1126,fc7707c0-2f1e-48d4-b870-7cbeddfc...,Philadelphia,2019-05-20 00:00:00.000000,true,true,true
1,2020,4,18,145,412,24888.19,2346,eafbd328-0434-4f46-9c2d-cc97a46f...,Brooklyn,2020-04-27 00:00:00.000000,true,true,true
2,2020,4,15,139,391,25648.41,2386,eafbd328-0434-4f46-9c2d-cc97a46f...,Brooklyn,2020-04-06 00:00:00.000000,true,true,true
3,2019,7,28,61,118,10507.1,1113,eafbd328-0434-4f46-9c2d-cc97a46f...,Brooklyn,2019-07-08 00:00:00.000000,true,true,true
4,2018,12,49,14,93,8689.4,737,fc7707c0-2f1e-48d4-b870-7cbeddfc...,Philadelphia,2018-12-03 00:00:00.000000,true,true,true
,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,2023,12,50,1162,1735,29879.13,2753,eafbd328-0434-4f46-9c2d-cc97a46f...,Brooklyn,2023-12-11 00:00:00.000000,true,true,true
1375,2024,5,22,1302,988,16341.42,1432,61743c9b-2394-4d36-8062-8a6820fa...,Los Angeles,2024-05-27 00:00:00.000000,true,true,true
1376,2024,6,24,1319,2109,18339.02,1571,fc7707c0-2f1e-48d4-b870-7cbeddfc...,Philadelphia,2024-06-10 00:00:00.000000,true,true,true


In [15]:

weekly_sales_by_store.set_role(["STORE_ID", "SNAPSHOT_ID"] , getml.data.roles.join_key)
weekly_sales_by_store.set_role("REFERENCE_DATE", getml.data.roles.time_stamp)
weekly_sales_by_store.set_role("NEXT_WEEK_SALES", getml.data.roles.target)
weekly_sales_by_store.set_role(["STORE_NAME", "YEAR", "MONTH", "WEEK_NUMBER", "IS_FULL_WEEK_AFTER_OPENING", "HAS_ORDER_ACTIVITY", "HAS_MIN_HISTORY"], getml.data.roles.categorical)
weekly_sales_by_store.set_role(["DAYS_SINCE_OPEN", "NEXT_WEEK_ORDERS"], getml.data.roles.numerical)


## getML Data Model

## Training

## Feature Export